# E-commerce ranking — offline evaluation of a candidate recommender

Mirror of `examples/use_cases/01_ecommerce_ranking.py`. Estimate the
policy value of a candidate ranker from logged session data, **before**
any A/B test.

For *slate* / top-K ranking estimators, see roadmap issue #75.

👉 [Open in Colab](https://colab.research.google.com/github/dgenio/skdr-eval/blob/main/examples/notebooks/03_ecommerce_ranking.ipynb)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet skdr-eval scikit-learn

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

import skdr_eval

logs, items, _ = skdr_eval.make_synth_logs(n=2000, n_ops=5, seed=7)
skdr_eval.validate_logs(logs)
list(items)

In [ ]:
candidates = {
    "RF_ranker": RandomForestRegressor(n_estimators=80, max_depth=8, random_state=7),
    "GBM_ranker": GradientBoostingRegressor(
        n_estimators=80, max_depth=4, random_state=7
    ),
}
artifact = skdr_eval.evaluate_sklearn_models(
    logs=logs,
    models=candidates,
    fit_models=True,
    n_splits=2,
    outcome_estimator="hgb",
    random_state=7,
    policy_train="pre_split",
    policy_train_frac=0.8,
)
cols = ["model", "estimator", "V_hat", "ESS", "match_rate", "support_health"]
artifact.report[cols].round(4)

Trust diagnostics:

In [ ]:
artifact.warnings

Pick the candidate with the best (lowest) DR estimate. In a real
deployment review, also confirm the support-health column is `ok` or
`caution`; `high_risk` means do **not** ship without further checks.

In [ ]:
dr_rows = artifact.report[artifact.report["estimator"] == "DR"]
best_name = dr_rows.loc[dr_rows["V_hat"].idxmin(), "model"]
best_name